# Dataset v2: more phrases and sentence frames

Builds `data/sarcasm_dataset_v2.csv` with `data/make_dataset_v2.py` (same design as v1, `data.ipynb`) and
describes it. **No GPU needed** (only the Qwen2.5 tokenizer).

| | v1 | v2 |
|---|---|---|
| situation pairs | 46 | 70 |
| positive phrases | 5 (1 held out) | 24 (6 held out) |
| sentence frames | 1 | 5 (1 held out) |
| triples | 230 | 910 (630 discovery, 210 held-out phrase, 70 held-out frame) |

Why: circuits found on v1 generalise to new situations but only partly to a new phrase, and poorly to free text.
v2 has far more variety in the wording, and held-out phrases **and** a held-out sentence frame.
All pairs are token-matched (same number of tokens, differences only in the changed part).

In [ ]:
import os, sys
REPO = "https://github.com/vasilis-pantzos/sarcasm-proposal.git"
ROOT = "/content/sarcasm-proposal"
if not os.path.exists(ROOT):
    !git clone -q {REPO} {ROOT}
else:
    !git -C {ROOT} pull -q
os.chdir(ROOT)
sys.path.insert(0, ROOT); sys.path.insert(0, os.path.join(ROOT, "data"))

## Build and compare with the CSV in the repository

In [ ]:
import pandas as pd
from transformers import AutoTokenizer
import make_dataset_v2 as m

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
df = m.build(tok)
if os.path.exists(m.CSV_PATH):
    old = pd.read_csv(m.CSV_PATH).fillna({"heldout_type": ""})
    same = old.drop(columns="heldout_type").equals(df.drop(columns="heldout_type"))
    print("same as the CSV in the repository:", same)
df.to_csv("sarcasm_dataset_v2_rebuilt.csv", index=False)
print(f"{len(df)} prompts, {df.triple_id.nunique()} triples")

## Counts

In [ ]:
t = df[df.condition == "S"].copy()
t["group"] = [s if s == "discovery" else f"heldout ({h})" for s, h in zip(t.split, t.heldout_type)]
display(t.groupby("group").agg(triples=("triple_id", "size"), token_matched_PS=("token_matched_PS", "sum"),
                               token_matched_SN=("token_matched_SN", "sum")))
display(t.groupby(["group", "context"]).size().rename("triples").to_frame())
display(t.groupby(["split", "positive_frame"]).size().rename("triples").to_frame())
print("prompt length (tokens):", df.n_tokens.min(), "-", df.n_tokens.max())

## Examples

In [ ]:
for tid in pd.Series(df.triple_id.unique()).sample(8, random_state=0):
    g = df[df.triple_id == tid].set_index("condition")
    print(f"--- triple {tid}  [{g.split.iloc[0]} {g.heldout_type.iloc[0]}]")
    for c in "PSN":
        print(f"  {c}: {g.loc[c, 'text']}")